In [7]:
import numpy as np
import h5py
import os
import cv2
import torch
import numpy as np
from numba import njit

def load_h5(filename):
    '''Loads dictionary from hdf5 file'''
    dict_to_load = {}
    if not os.path.isfile(filename):
        print('Cannot find file {}'.format(filename))
        return None
    with h5py.File(filename, 'r') as f:
        keys = [key for key in f.keys()]
        for key in keys:
            dict_to_load[key] = f[key][()]
    return dict_to_load

def load_h5_nested(filename):
    '''Loads dictionary from hdf5 file'''
    dict_to_load = {}
    if not os.path.isfile(filename):
        print('Cannot find file {}'.format(filename))
        return None
    dict_to_load= {}
    with h5py.File(filename, 'r') as f:
        keys = [key for key in f.keys()]
        for key in keys:
            dd = f[key]
            for k in dd.keys():
                matches = dd[k][()]
                final_key = f'{key}-{k}'
                dict_to_load[final_key] = matches
    return dict_to_load


In [2]:
roma_dir ='/home/dmytromishkin/big_storage/ransac-tutorial2/data/ransac2020/roma-feats/val/sacre_coeur_roma'
matches = load_h5_nested(f'{roma_dir}/matches_roma_800.h5')
keys = [k for k in matches.keys()]

In [3]:
F1 = np.array([[0,0, 0],
               [0,0,-1],
               [0, 1, 0]])

In [4]:
import numpy as np

def _to_homogeneous(pts: np.ndarray) -> np.ndarray:
    # pts: (..., N, 2) -> (..., N, 3) by appending 1s
    ones = np.ones((*pts.shape[:-1], 1), dtype=pts.dtype)
    return np.concatenate([pts, ones], axis=-1)

def sampson_epipolar_distance_numpy(
    pts1: np.ndarray,
    pts2: np.ndarray,
    Fm: np.ndarray,
    squared: bool = True,
    eps: float = 1e-8,
) -> np.ndarray:
    """
    NumPy version of Sampson epipolar distance.

    pts1: (..., N, 2|3)
    pts2: (..., N, 2|3)
    Fm:   (..., 3, 3)
    returns: (..., N)
    """

    if pts1.shape[-1] == 2:
        pts1 = _to_homogeneous(pts1)

    if pts2.shape[-1] == 2:
        pts2 = _to_homogeneous(pts2)

    # F^T
    F_t = np.swapaxes(Fm, -2, -1)

    # lines
    # line1_in_2 = pts1 @ F^T
    # line2_in_1 = pts2 @ F
    line1_in_2 = pts1 @ F_t
    line2_in_1 = pts2 @ Fm

    # numerator = (x'^T F x)^2
    numerator = np.sum(pts2 * line1_in_2, axis=-1) ** 2

    # denominator = ||(Fx)_(1:2)||^2 + ||(F^T x')_(1:2)||^2
    # which is squared L2 norm of first two coords
    denom1 = np.sum(line1_in_2[..., :2] ** 2, axis=-1)
    denom2 = np.sum(line2_in_1[..., :2] ** 2, axis=-1)
    denominator = denom1 + denom2

    out = numerator / denominator

    if squared:
        return out
    return np.sqrt(out + eps)


Help on built-in function sampsonDistance:

sampsonDistance(...)
    sampsonDistance(pt1, pt2, F) -> retval
    .   @brief Calculates the Sampson Distance between two points.
    .   
    .   The function cv::sampsonDistance calculates and returns the first order approximation of the geometric error as:
    .   \f[
    .   sd( \texttt{pt1} , \texttt{pt2} )=
    .   \frac{(\texttt{pt2}^t \cdot \texttt{F} \cdot \texttt{pt1})^2}
    .   {((\texttt{F} \cdot \texttt{pt1})(0))^2 +
    .   ((\texttt{F} \cdot \texttt{pt1})(1))^2 +
    .   ((\texttt{F}^t \cdot \texttt{pt2})(0))^2 +
    .   ((\texttt{F}^t \cdot \texttt{pt2})(1))^2}
    .   \f]
    .   The fundamental matrix may be calculated using the #findFundamentalMat function. See @cite HartleyZ00 11.4.3 for details.
    .   @param pt1 first homogeneous 2d point
    .   @param pt2 second homogeneous 2d point
    .   @param F fundamental matrix
    .   @return The computed Sampson distance.



In [8]:
@njit(cache=True, fastmath=True)
def sampson_residuals_sq_numba(F: np.ndarray, src: np.ndarray, dst: np.ndarray, out_sq: np.ndarray):
    N = src.shape[0]
    for i in range(N):
        x, y = src[i,0], src[i,1]
        u, v = dst[i,0], dst[i,1]
        Fx0 = F[0,0]*x + F[0,1]*y + F[0,2]
        Fx1 = F[1,0]*x + F[1,1]*y + F[1,2]
        Fx2 = F[2,0]*x + F[2,1]*y + F[2,2]
        Ft0 = F[0,0]*u + F[1,0]*v + F[2,0]
        Ft1 = F[0,1]*u + F[1,1]*v + F[2,1]
        num = u*Fx0 + v*Fx1 + Fx2
        den = Fx0*Fx0 + Fx1*Fx1 + Ft0*Ft0 + Ft1*Ft1 + 1e-15
        out_sq[i] = (num*num) / den

In [9]:
def sampson_residuals_sq_numpy(F, src, dst):
    # src: (N,2) -> x,y
    # dst: (N,2) -> u,v
    x = src[:, 0]
    y = src[:, 1]
    u = dst[:, 0]
    v = dst[:, 1]

    Fx0 = F[0,0]*x + F[0,1]*y + F[0,2]
    Fx1 = F[1,0]*x + F[1,1]*y + F[1,2]
    Fx2 = F[2,0]*x + F[2,1]*y + F[2,2]

    Ft0 = F[0,0]*u + F[1,0]*v + F[2,0]
    Ft1 = F[0,1]*u + F[1,1]*v + F[2,1]

    num = u*Fx0 + v*Fx1 + Fx2
    den = Fx0*Fx0 + Fx1*Fx1 + Ft0*Ft0 + Ft1*Ft1 + 1e-15

    return (num*num) / den

In [10]:
def sampson_residuals_sq_torch(F, src, dst):
    # src: (N,2) -> x,y
    # dst: (N,2) -> u,v
    x = src[:, 0]
    y = src[:, 1]
    u = dst[:, 0]
    v = dst[:, 1]

    Fx0 = F[0,0]*x + F[0,1]*y + F[0,2]
    Fx1 = F[1,0]*x + F[1,1]*y + F[1,2]
    Fx2 = F[2,0]*x + F[2,1]*y + F[2,2]

    Ft0 = F[0,0]*u + F[1,0]*v + F[2,0]
    Ft1 = F[0,1]*u + F[1,1]*v + F[2,1]

    num = u*Fx0 + v*Fx1 + Fx2
    den = Fx0*Fx0 + Fx1*Fx1 + Ft0*Ft0 + Ft1*Ft1 + 1e-15

    return (num*num) / den

In [22]:
matches_single = matches[keys[0]]
matches_single.shape
src, dst = torch.from_numpy(matches_single[:,:2]), torch.from_numpy(matches_single[:,2:])

In [23]:
@njit(cache=True, fastmath=True)
def sampson_residuals_sq_numba(F: np.ndarray, src: np.ndarray, dst: np.ndarray, out_sq: np.ndarray):
    N = src.shape[0]
    for i in range(N):
        x, y = src[i,0], src[i,1]
        u, v = dst[i,0], dst[i,1]
        Fx0 = F[0,0]*x + F[0,1]*y + F[0,2]
        Fx1 = F[1,0]*x + F[1,1]*y + F[1,2]
        Fx2 = F[2,0]*x + F[2,1]*y + F[2,2]
        Ft0 = F[0,0]*u + F[1,0]*v + F[2,0]
        Ft1 = F[0,1]*u + F[1,1]*v + F[2,1]
        num = u*Fx0 + v*Fx1 + Fx2
        den = Fx0*Fx0 + Fx1*Fx1 + Ft0*Ft0 + Ft1*Ft1 + 1e-15
        out_sq[i] = (num*num) / den

In [13]:
out = np.zeros_like(matches_single[:,0])
sampson_residuals_sq_numba(F1, matches_single[:,:2], matches_single[:,2:], out)

In [ ]:
%timeit dist = sampson_epipolar_distance_numpy(matches_single[:,:2], matches_single[:,2:], F1, True)

In [14]:
src1 = np.ascontiguousarray(matches_single[:,:2])
dst1 = np.ascontiguousarray(matches_single[:,2:4])

%timeit dist = sampson_residuals_sq_numpy(F1, src1, dst1)

294 μs ± 528 ns per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [ ]:
%timeit sampson_residuals_sq_numba(F1, matches_single[:,:2], matches_single[:,2:], out)

In [ ]:
%timeit with torch.inference_mode(): dist = sampson_residuals_sq_torch(torch.from_numpy(F1).float(), src.float(), dst.float())

In [15]:
def sampson_residuals_sq_numpy_hoist(F, src, dst):
    f00, f01, f02 = F[0,0], F[0,1], F[0,2]
    f10, f11, f12 = F[1,0], F[1,1], F[1,2]
    f20, f21, f22 = F[2,0], F[2,1], F[2,2]

    x = src[:, 0]
    y = src[:, 1]
    u = dst[:, 0]
    v = dst[:, 1]

    Fx0 = f00*x + f01*y + f02
    Fx1 = f10*x + f11*y + f12
    Fx2 = f20*x + f21*y + f22

    Ft0 = f00*u + f10*v + f20
    Ft1 = f01*u + f11*v + f21

    num = u*Fx0 + v*Fx1 + Fx2
    den = Fx0*Fx0 + Fx1*Fx1 + Ft0*Ft0 + Ft1*Ft1 + 1e-15

    return (num*num) / den


In [ ]:
%timeit dist = sampson_residuals_sq_numpy_hoist(F1, src1, dst1)

In [16]:
def sampson_residuals_sq_numpy_inplace(F, src, dst):
    f00, f01, f02 = F[0,0], F[0,1], F[0,2]
    f10, f11, f12 = F[1,0], F[1,1], F[1,2]
    f20, f21, f22 = F[2,0], F[2,1], F[2,2]

    x = src[:, 0]
    y = src[:, 1]
    u = dst[:, 0]
    v = dst[:, 1]

    # We'll allocate a couple temp arrays of length N and reuse them.

    # Fx0 = f00*x + f01*y + f02
    Fx0 = f00 * x
    Fx0 += f01 * y
    Fx0 += f02

    # Fx1 = f10*x + f11*y + f12
    Fx1 = f10 * x
    Fx1 += f11 * y
    Fx1 += f12

    # Fx2 = f20*x + f21*y + f22
    Fx2 = f20 * x
    Fx2 += f21 * y
    Fx2 += f22

    # Ft0 = f00*u + f10*v + f20
    Ft0 = f00 * u
    Ft0 += f10 * v
    Ft0 += f20

    # Ft1 = f01*u + f11*v + f21
    Ft1 = f01 * u
    Ft1 += f11 * v
    Ft1 += f21

    # num = u*Fx0 + v*Fx1 + Fx2
    num = u * Fx0
    num += v * Fx1
    num += Fx2

    # den = Fx0^2 + Fx1^2 + Ft0^2 + Ft1^2 + eps
    den = Fx0 * Fx0
    den += Fx1 * Fx1
    den += Ft0 * Ft0
    den += Ft1 * Ft1
    den += 1e-15

    out = num * num
    out /= den
    return out


In [ ]:
%timeit dist = sampson_residuals_sq_numpy_inplace(F1, src1, dst1)

In [24]:
src_c, dst_c, Fc = src.cuda().float(), dst.cuda().float(), torch.from_numpy(F1).cuda().float()

In [25]:
def samp_cuda():
    torch.cuda.synchronize()
    with torch.inference_mode():
        out = sampson_residuals_sq_torch(Fc, src_c, dst_c)
    torch.cuda.synchronize()
    return out

In [26]:
def samp_cpu_torch():
    with torch.inference_mode():
        out = sampson_residuals_sq_numpy_inplace(Fcpu, src, dst)
    return out

In [27]:
def samp_cuda2():
    torch.cuda.synchronize()
    with torch.inference_mode():
        out = sampson_residuals_sq_numpy_inplace(Fc, src_c, dst_c)
    torch.cuda.synchronize()
    return out

In [31]:
def sampson_residuals_sq_numpy(F, src, dst):
    # src: (N,2) -> x,y
    # dst: (N,2) -> u,v
    x = src[:, 0]
    y = src[:, 1]
    u = dst[:, 0]
    v = dst[:, 1]

    Fx0 = F[0,0]*x + F[0,1]*y + F[0,2]
    Fx1 = F[1,0]*x + F[1,1]*y + F[1,2]
    Fx2 = F[2,0]*x + F[2,1]*y + F[2,2]

    Ft0 = F[0,0]*u + F[1,0]*v + F[2,0]
    Ft1 = F[0,1]*u + F[1,1]*v + F[2,1]

    num = u*Fx0 + v*Fx1 + Fx2
    den = Fx0*Fx0 + Fx1*Fx1 + Ft0*Ft0 + Ft1*Ft1 + 1e-15

    return (num*num) / den

def sampson_residuals_sq_numpy_inplace(F, src, dst):
    f00, f01, f02 = F[0,0], F[0,1], F[0,2]
    f10, f11, f12 = F[1,0], F[1,1], F[1,2]
    f20, f21, f22 = F[2,0], F[2,1], F[2,2]

    x = src[:, 0]
    y = src[:, 1]
    u = dst[:, 0]
    v = dst[:, 1]

    # We'll allocate a couple temp arrays of length N and reuse them.

    # Fx0 = f00*x + f01*y + f02
    Fx0 = f00 * x
    Fx0 += f01 * y
    Fx0 += f02

    # Fx1 = f10*x + f11*y + f12
    Fx1 = f10 * x
    Fx1 += f11 * y
    Fx1 += f12

    # Fx2 = f20*x + f21*y + f22
    Fx2 = f20 * x
    Fx2 += f21 * y
    Fx2 += f22

    # Ft0 = f00*u + f10*v + f20
    Ft0 = f00 * u
    Ft0 += f10 * v
    Ft0 += f20

    # Ft1 = f01*u + f11*v + f21
    Ft1 = f01 * u
    Ft1 += f11 * v
    Ft1 += f21

    # num = u*Fx0 + v*Fx1 + Fx2
    num = u * Fx0
    num += v * Fx1
    num += Fx2

    # den = Fx0^2 + Fx1^2 + Ft0^2 + Ft1^2 + eps
    den = Fx0 * Fx0
    den += Fx1 * Fx1
    den += Ft0 * Ft0
    den += Ft1 * Ft1
    den += 1e-15

    out = num * num
    out /= den
    return out


Fcpu = torch.from_numpy(F1)
src1 = np.ascontiguousarray(matches_single[:,:2])
dst1 = np.ascontiguousarray(matches_single[:,2:4])


src_t = torch.from_numpy(src1)
dst_t = torch.from_numpy(dst1)

def samp_cpu_first_torch():
    with torch.inference_mode():
        out = sampson_residuals_sq_numpy(Fcpu, src_t, dst_t)
    return out
def samp_cpu_first_numpy():
    out = sampson_residuals_sq_numpy(F1, src1, dst1)
    return out
print ("Pytorch CPU")
%timeit dist = samp_cpu_first_torch()
print ("Numpy CPU")
%timeit dist = samp_cpu_first_numpy()


def samp_cpu_first_torch_opt():
    with torch.inference_mode():
        out = sampson_residuals_sq_numpy_inplace(Fcpu, src_t, dst_t)
    return out
def samp_cpu_first_numpy_opt():
    out = sampson_residuals_sq_numpy_inplace(F1, src1, dst1)
    return out
print ("Pytorch CPU inplace")
%timeit dist = samp_cpu_first_torch_opt()
print ("Numpy CPU inplace")
%timeit dist = samp_cpu_first_numpy_opt()


Pytorch CPU
375 μs ± 546 ns per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
Numpy CPU
297 μs ± 414 ns per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
Pytorch CPU inplace
299 μs ± 581 ns per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
Numpy CPU inplace
272 μs ± 382 ns per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [60]:
%timeit dist = samp_cpu_torch()

296 μs ± 8.38 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [29]:
src_c, dst_c, Fc = src.cuda().float(), dst.cuda().float(), torch.from_numpy(F1).cuda().float()
torch.cuda.synchronize()
def samp_cuda():
    torch.cuda.synchronize()
    with torch.inference_mode():
        out = sampson_residuals_sq_numpy(Fc, src_c, dst_c)
    torch.cuda.synchronize()
    return out

def samp_cuda_opt():
    torch.cuda.synchronize()
    with torch.inference_mode():
        out = sampson_residuals_sq_numpy_inplace(Fc, src_c, dst_c)
    torch.cuda.synchronize()
    return out
print ("Pytorch cuda")
%timeit dist = samp_cuda()

print ("Pytorch cuda opt")
%timeit dist = samp_cuda_opt()

Pytorch cuda
501 μs ± 58.7 μs per loop (mean ± std. dev. of 7 runs, 1 loop each)
Pytorch cuda opt
367 μs ± 974 ns per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [34]:
import torch

@torch.compile  # uses TorchInductor to fuse kernels
def sampson_residuals_sq_torch_compiled(F, src, dst):
    # F: (3,3) cuda float32
    # src, dst: (N,2) cuda float32
    x = src[:, 0]
    y = src[:, 1]
    u = dst[:, 0]
    v = dst[:, 1]

    f00, f01, f02 = F[0,0], F[0,1], F[0,2]
    f10, f11, f12 = F[1,0], F[1,1], F[1,2]
    f20, f21, f22 = F[2,0], F[2,1], F[2,2]

    Fx0 = f00 * x + f01 * y + f02
    Fx1 = f10 * x + f11 * y + f12
    Fx2 = f20 * x + f21 * y + f22

    Ft0 = f00 * u + f10 * v + f20
    Ft1 = f01 * u + f11 * v + f21

    num = u * Fx0 + v * Fx1 + Fx2
    den = Fx0*Fx0 + Fx1*Fx1 + Ft0*Ft0 + Ft1*Ft1 + 1e-15

    return (num * num) / den


def samp_cuda_comp():
    torch.cuda.synchronize()
    with torch.inference_mode():
        out = sampson_residuals_sq_torch_compiled(Fc, src_c, dst_c)
    torch.cuda.synchronize()
    return out
print ("Pytorch cuda comp")
dist = samp_cuda_comp()
%timeit dist = samp_cuda_comp()

Pytorch cuda comp
196 μs ± 2.33 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [35]:
%timeit dist = samp_cuda()

449 μs ± 1.39 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [57]:
%timeit dist = samp_cuda2()

348 μs ± 906 ns per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [ ]:
%timeit dist = samp_cuda()

In [36]:
import numpy as np
from numba import cuda, float32

@cuda.jit
def sampson_residuals_sq_cuda(F, src, dst, out_sq):
    i = cuda.grid(1)
    N = src.shape[0]
    if i < N:
        x = src[i, 0]
        y = src[i, 1]
        u = dst[i, 0]
        v = dst[i, 1]

        f00 = F[0, 0]; f01 = F[0, 1]; f02 = F[0, 2]
        f10 = F[1, 0]; f11 = F[1, 1]; f12 = F[1, 2]
        f20 = F[2, 0]; f21 = F[2, 1]; f22 = F[2, 2]

        Fx0 = f00*x + f01*y + f02
        Fx1 = f10*x + f11*y + f12
        Fx2 = f20*x + f21*y + f22

        Ft0 = f00*u + f10*v + f20
        Ft1 = f01*u + f11*v + f21

        num = u*Fx0 + v*Fx1 + Fx2
        den = Fx0*Fx0 + Fx1*Fx1 + Ft0*Ft0 + Ft1*Ft1 + 1e-15

        out_sq[i] = (num*num) / den


In [37]:
def sampson_residuals_sq_numba_cuda_driver(F, src, dst):
    # Ensure float32 contiguous
    F32   = np.ascontiguousarray(F.astype(np.float32))
    src32 = np.ascontiguousarray(src.astype(np.float32))
    dst32 = np.ascontiguousarray(dst.astype(np.float32))

    N = src32.shape[0]

    # Allocate output
    out_sq = np.empty(N, dtype=np.float32)

    # Copy to device
    d_F    = cuda.to_device(F32)
    d_src  = cuda.to_device(src32)
    d_dst  = cuda.to_device(dst32)
    d_out  = cuda.device_array(N, dtype=np.float32)

    # Configure launch
    threads_per_block = 256
    blocks = (N + threads_per_block - 1) // threads_per_block

    # Warmup / launch
    sampson_residuals_sq_cuda[blocks, threads_per_block](d_F, d_src, d_dst, d_out)

    # Copy back
    d_out.copy_to_host(out_sq)

    return out_sq


In [38]:
def prepare_cuda_buffers(F, src, dst):
    F32   = np.ascontiguousarray(F.astype(np.float32))
    src32 = np.ascontiguousarray(src.astype(np.float32))
    dst32 = np.ascontiguousarray(dst.astype(np.float32))

    d_F    = cuda.to_device(F32)
    d_src  = cuda.to_device(src32)
    d_dst  = cuda.to_device(dst32)
    d_out  = cuda.device_array(src32.shape[0], dtype=np.float32)

    return d_F, d_src, d_dst, d_out

def run_cuda_kernel(d_F, d_src, d_dst, d_out):
    N = d_src.shape[0]
    threads_per_block = 256
    blocks = (N + threads_per_block - 1) // threads_per_block

    sampson_residuals_sq_cuda[blocks, threads_per_block](d_F, d_src, d_dst, d_out)

    # synchronize for timing fairness
    cuda.synchronize()

def fetch_result(d_out):
    return d_out.copy_to_host()


In [39]:
%timeit dist = sampson_residuals_sq_numba_cuda_driver(F1, src1, dst1)

/home/dmytromishkin/big_storage/miniconda/envs/py310/lib/python3.10/site-packages/numba/cuda/dispatcher.py:536: NumbaPerformanceWarning: Grid size 40 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))


1.23 ms ± 125 μs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [40]:
d_F, d_src, d_dst, d_out = prepare_cuda_buffers(F1, src1, dst1)
%timeit dist = run_cuda_kernel(d_F, d_src, d_dst, d_out)

80.8 μs ± 230 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
